# Results for model: mistralai/mistral-7b-instruct-v0.3

In [1]:
import polars as pl
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Load data
data = pl.read_parquet('./jane_street_data/train.parquet')

# Feature engineering
batch_size = int(data.shape[0] * 0.8)
rolling_window = pl.Window().every(batch_size).over('date_id').discarding(pl.UDF(lambda w: w.last() - w.start(), pl.Int64)).alias('window')

top_quartile = data.select([
    pl.col('feature_00'),
    pl.col('responder_6').alias('y')
]) \
    .groupby('date_id', 'window') \
    .agg({'feature_00': pl.mean(), 'y': pl.mean()}) \
    .sort('date_id', ascending=True) \
    .drop_duplicates(must_contain='date_id') \
    .head(int(len(data) * 0.15))['feature_00'] \
    .as_series().values[0]

# Filter out the top quantile from the 'feature_00' column
filtered_data = data.filter(pl.col('feature_00') < top_quartile)

# Prepare training and testing datasets
X = filtered_data.drop('y')
y = filtered_data['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train XGBoost Regressor
model = XGBRegressor(n_jobs=-1)
model.fit(X_train, y_train)

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet